# Day 3 · 실습 04. CLIP Zero-shot 이미지 분류

**연계 이론:** 이론 4(CLIP과 Vision-Language Model)

CLIP ViT-B/32로 Image/Text Embedding을 추출하고 Cosine Similarity를 비교합니다.
개미·벌 후보 텍스트 중 유사도가 높은 클래스를 선택합니다.
**프롬프트 A·B·C 변경 비교 실험은 포함하지 않습니다.**

**사전 준비:** OpenAI 공식 CLIP 설치
`python -m pip install git+https://github.com/openai/CLIP.git`
필요 패키지: torch, torchvision, pillow, numpy, pandas, matplotlib, ftfy, regex, tqdm
공식 저장소: https://github.com/openai/CLIP

In [ ]:
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

try:
    import clip
except ImportError as exc:
    raise ImportError(
        "OpenAI 공식 CLIP 설치가 필요합니다: "
        "python -m pip install git+https://github.com/openai/CLIP.git"
    ) from exc
if not hasattr(clip, "load") or not hasattr(clip, "tokenize"):
    raise ImportError("현재 clip 모듈은 OpenAI 공식 CLIP이 아닙니다. 설치를 확인하세요.")

SEED = 42
rng = np.random.default_rng(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_ROOT = Path(os.getenv("DAY3_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
ASSETS_DIR = PROJECT_ROOT / "assets"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_ROOT = Path(os.getenv("DAY3_DATA_ROOT", str(ASSETS_DIR / "hymenoptera_data"))).expanduser().resolve()
CLIP_MODEL_SOURCE = os.getenv("DAY3_CLIP_MODEL", "ViT-B/32")
CLIP_CACHE_DIR = Path(os.getenv("DAY3_CLIP_CACHE_DIR", str(WEIGHTS_DIR / "clip"))).expanduser().resolve()
for folder in (ASSETS_DIR, WEIGHTS_DIR, OUTPUTS_DIR, CLIP_CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| CLIP:", CLIP_MODEL_SOURCE)
print("데이터:", DATA_ROOT)

## 1. 데이터셋 준비

개미·벌 검증 이미지 12장을 사용합니다. 저장된 데이터가 있으면 재사용합니다.
데이터 정답은 정확도 확인용이며 CLIP의 추가 학습에 사용하지 않습니다.

In [ ]:
DATA_URL = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

def ensure_dataset(root=DATA_ROOT):
    required = [root / split / name for split in ("train", "val") for name in ("ants", "bees")]
    if all(p.is_dir() for p in required):
        print("기존 데이터 사용:", root)
        return root

    root.parent.mkdir(parents=True, exist_ok=True)
    archive = root.parent / "hymenoptera_data.zip"
    if not archive.exists():
        print("개미·벌 데이터 다운로드 중...")
        temp = archive.with_suffix(".zip.part")
        try:
            with urllib.request.urlopen(DATA_URL, timeout=90) as response, temp.open("wb") as out:
                shutil.copyfileobj(response, out)
            temp.replace(archive)
        finally:
            temp.unlink(missing_ok=True)

    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            parts = Path(member.filename).parts
            if not parts or parts[0] != "hymenoptera_data":
                raise ValueError("예상하지 못한 ZIP 경로: " + member.filename)
            if len(parts) == 1:
                continue
            destination = (root / Path(*parts[1:])).resolve()
            if not destination.is_relative_to(root.resolve()):
                raise ValueError("허용되지 않은 ZIP 경로: " + member.filename)
            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, destination.open("wb") as dst:
                    shutil.copyfileobj(src, dst)

    if not all(p.is_dir() for p in required):
        raise FileNotFoundError(f"train/val/ants/bees 폴더를 확인하세요: {root}")
    print("데이터 준비 완료:", root)
    return root

ensure_dataset()

In [ ]:
IMAGES_PER_CLASS = 6
image_paths, true_labels = [], []
for class_name in ("ants", "bees"):
    candidates = sorted((DATA_ROOT / "val" / class_name).glob("*.jpg"))
    if len(candidates) < IMAGES_PER_CLASS:
        raise ValueError(f"{class_name}: 이미지가 부족합니다.")
    chosen = rng.choice(len(candidates), size=IMAGES_PER_CLASS, replace=False)
    for idx in chosen:
        image_paths.append(candidates[int(idx)])
        true_labels.append(class_name)

images = []
for path in image_paths:
    with Image.open(path) as im:
        images.append(im.convert("RGB"))
short_labels = [f"{name}-{i+1}" for name in ("ants", "bees")
                for i in range(IMAGES_PER_CLASS)]

fig, axes = plt.subplots(2, IMAGES_PER_CLASS, figsize=(15, 5))
for ax, im, label in zip(axes.flat, images, short_labels):
    ax.imshow(im)
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. CLIP ViT-B/32 불러오기

`clip.load()`는 모델과 해당 모델에 맞는 전처리 함수를 함께 반환합니다.
DINOv3의 전처리 설정을 CLIP에 그대로 적용하지 않습니다.

In [ ]:
try:
    model, preprocess = clip.load(
        CLIP_MODEL_SOURCE, device=DEVICE, jit=False, download_root=str(CLIP_CACHE_DIR)
    )
except Exception as exc:
    raise RuntimeError(
        "CLIP 로딩 실패: 공식 패키지 설치, 네트워크 연결, 모델 경로를 확인하세요."
    ) from exc
model.eval()
image_inputs = torch.stack([preprocess(im) for im in images]).to(DEVICE)
print("이미지 입력 Shape:", tuple(image_inputs.shape))

## 3. Image Embedding과 Text Embedding

미리 정한 두 후보 텍스트만 사용합니다. 새로운 Head를 학습하지 않습니다.
텍스트 후보와 클래스 순서는 서로 대응해야 합니다.

In [ ]:
CLASS_NAMES = ["ants", "bees"]
CLASS_TEXTS = ["a photo of an ant", "a photo of a bee"]

with torch.inference_mode():
    image_vectors = F.normalize(model.encode_image(image_inputs).float(), dim=-1)
    tokens = clip.tokenize(CLASS_TEXTS).to(DEVICE)
    text_vectors = F.normalize(model.encode_text(tokens).float(), dim=-1)

print("Image Embeddings:", tuple(image_vectors.shape))
print("Text Embeddings:", tuple(text_vectors.shape))
print("후보 텍스트:", CLASS_TEXTS)

## 4. Zero-shot Classification

L2 정규화한 Image/Text Embedding의 내적으로 Cosine Similarity를 계산합니다.
각 이미지에서 유사도가 더 높은 후보 클래스를 예측으로 선택합니다.

In [ ]:
cosine_scores = (image_vectors @ text_vectors.T).cpu().numpy()
pred_indices = cosine_scores.argmax(axis=1)
pred_labels = [CLASS_NAMES[i] for i in pred_indices]

prediction_df = pd.DataFrame({
    "image": short_labels,
    "true_label": true_labels,
    "cos_ant": cosine_scores[:, 0],
    "cos_bee": cosine_scores[:, 1],
    "prediction": pred_labels,
})
prediction_df["correct"] = prediction_df["true_label"] == prediction_df["prediction"]
print(prediction_df.round(3).to_string(index=False))
print("예시 이미지 12장 정확도:", f"{prediction_df['correct'].mean():.1%}")
print("※ 작은 검증 샘플의 결과이며 일반적인 모델 성능을 의미하지 않습니다.")

fig, ax = plt.subplots(figsize=(7, 5))
heatmap = ax.imshow(cosine_scores, vmin=-1, vmax=1, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(CLASS_TEXTS)), CLASS_TEXTS, rotation=10)
ax.set_yticks(range(len(short_labels)), short_labels)
ax.set_title("CLIP Image-Text Cosine Similarity")
fig.colorbar(heatmap, ax=ax, label="Cosine Similarity")
plt.tight_layout()
plt.show()

## 5. Softmax 상대 점수 확인

CLIP의 학습된 `logit_scale`을 적용한 뒤 Softmax를 계산합니다.
이 값은 현재 두 후보 사이의 상대 점수이며 보정된 실제 정답 확률은 아닙니다.

In [ ]:
with torch.inference_mode():
    logits = model.logit_scale.exp() * (image_vectors @ text_vectors.T)
    relative_scores = logits.softmax(dim=-1).cpu().numpy()
prediction_df["score_ant"] = relative_scores[:, 0]
prediction_df["score_bee"] = relative_scores[:, 1]

print("첫 번째 이미지:", short_labels[0])
for label, score in zip(CLASS_NAMES, relative_scores[0]):
    print(f"{label}: 상대 점수 {score:.3f}")
print("※ Softmax 결과를 실제 정답 확률로 해석하지 않습니다.")

## 6. 결과 저장 및 요약

- CLIP은 Image Encoder와 Text Encoder의 Embedding 유사도로 분류했습니다.
- 이번 실습에서 새로운 Classification Head나 가중치 학습은 없었습니다.
- Cosine Similarity와 Softmax 상대 점수는 구분해서 해석해야 합니다.

In [ ]:
prediction_df.to_csv(
    OUTPUTS_DIR / "day3_04_clip_zero_shot_predictions.csv",
    index=False, encoding="utf-8-sig"
)
print("결과 저장:", OUTPUTS_DIR)